# On-Device AI 시험 대비 연습 문제

각 문제에서 `## Write Answer` 부분을 채워 넣으세요.

**출제 범위**
- Part 1. Pruning (문제 1-7)
- Part 2. Quantization (문제 8-13)
- Part 3. Knowledge Distillation (문제 14-16)

# Part 1. Pruning

가중치의 중요도(importance)를 판단하는 기준을 이해하고, 이를 통해 불필요한 연산을 제거하는 과정을 다루는 문제입니다.

## 문제 1. Fine-grained Magnitude Pruning

In [ ]:
def prune_weight_fine_grained(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    num_pruned_elements = ## Write Answer
    importance = ## Write Answer
    threshold = ## Write Answer
    mask = ## Write Answer

    weight.mul_(mask)
    return mask

## 문제 2. Channel-level Pruning

In [ ]:
def prune_weight_channel_level(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    num_channels = ## Write Answer
    num_pruned_channels = ## Write Answer
    importance = ## Write Answer

    threshold = torch.kthvalue(importance.flatten(), num_pruned_channels)[0]
    mask = importance > threshold
    mask = mask.expand_as(weight)
    weight.mul_(mask)
    return mask

## 문제 3. Kernel-level Pruning

In [ ]:
def prune_weight_kernel_level(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    num_kernels = ## Write Answer
    num_pruned_kernels = ## Write Answer
    importance = ## Write Answer

    threshold = torch.kthvalue(importance.flatten(), num_pruned_kernels)[0]
    mask = importance > threshold
    mask = mask.expand_as(weight)
    weight.mul_(mask)
    return mask

## 문제 4. Global Magnitude Pruning

In [ ]:
def global_prune(model, sparsity):
    parameters_to_prune = []
    for name, param in model.named_parameters():
        if param.dim() > 1:
            parameters_to_prune.append(param.view(-1))

    all_weights = ## Write Answer
    num_elements = all_weights.numel()
    num_zeros = ## Write Answer
    importance = ## Write Answer
    threshold = ## Write Answer

    masks = {}
    for name, param in model.named_parameters():
        if param.dim() > 1:
            masks[name] = torch.abs(param.data) > threshold
    return masks

## 문제 5. LLM Magnitude Pruning

In [ ]:
@torch.no_grad()
def prune_magnitude(model, sparsity):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            num_elements = ## Write Answer
            num_zeros = ## Write Answer
            importance = ## Write Answer
            threshold = ## Write Answer
            mask = ## Write Answer
            W.mul_(mask)

## 문제 6. Wanda용 Activation L2 Norm 누적

In [ ]:
def compute_activation_norm(x):
    x = x.type(torch.float32)
    activation_norm = ## Write Answer
    return activation_norm

## 문제 7. Wanda Pruning

In [ ]:
@torch.no_grad()
def prune_wanda(model, sparsity, input_feat):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            row, col = W.shape
            num_zeros_per_row = ## Write Answer
            importance = ## Write Answer
            threshold = ## Write Answer
            mask = ## Write Answer
            W.mul_(mask)

# Part 2. Quantization

모델을 layer 단위에서 양자화/역양자화 하는 과정을 다루는 문제입니다.

## 문제 8. Linear Quantization (FP → INT)

In [ ]:
def linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype=torch.int8):
    scaled_tensor = ## Write Answer
    rounded_tensor = ## Write Answer
    rounded_tensor = rounded_tensor.to(dtype)
    shifted_tensor = ## Write Answer
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    quantized_tensor = shifted_tensor.clamp_(quantized_min, quantized_max)
    return quantized_tensor

## 문제 9. Quantization Scale & Zero Point 계산

In [ ]:
def get_quantization_scale_and_zero_point(fp_tensor, bitwidth):
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    fp_max = fp_tensor.max().item()
    fp_min = fp_tensor.min().item()

    scale = ## Write Answer
    zero_point = ## Write Answer

    if zero_point < quantized_min:
        zero_point = quantized_min
    elif zero_point > quantized_max:
        zero_point = quantized_max
    else:
        zero_point = round(zero_point)
    return scale, int(zero_point)

## 문제 10. Quantized Linear: Output 재스케일 & Zero Point 보정

In [ ]:
def quantized_linear_core(output, input_scale, weight_scale, output_scale, output_zero_point):
    weight_scale = weight_scale.flatten().view(1, -1)
    output = output.float()

    output = ## Write Answer
    output = ## Write Answer

    return output

## 문제 11. Quantized Conv2d: Output 재스케일 & Zero Point 보정

In [ ]:
def quantized_conv2d_core(output, input_scale, weight_scale, output_scale, output_zero_point):
    weight_scale = weight_scale.flatten().view(1, -1, 1, 1)
    output = output.float()

    output = ## Write Answer
    output = ## Write Answer

    return output

## 문제 12. K-means Quantization

In [ ]:
def k_means_quantize(fp32_tensor, bitwidth=4, codebook=None):
    if codebook is None:
        n_clusters = ## Write Answer
        kmeans = KMeans(n_clusters=n_clusters, mode='euclidean', verbose=0)
        labels = kmeans.fit_predict(fp32_tensor.view(-1, 1)).to(torch.long)
        centroids = kmeans.centroids.to(torch.float).view(-1)
        codebook = Codebook(centroids, labels)

    quantized_tensor = ## Write Answer
    fp32_tensor.set_(quantized_tensor.view_as(fp32_tensor))
    return codebook

## 문제 13. K-means Codebook 업데이트

In [ ]:
def update_codebook(fp32_tensor, codebook):
    n_clusters = codebook.centroids.numel()
    fp32_tensor = fp32_tensor.view(-1)
    for k in range(n_clusters):
        codebook.centroids[k] = ## Write Answer

# Part 3. Knowledge Distillation

지식증류의 원리를 이해하고, Teacher 모델의 예측값과 정답값을 활용해서 Student 모델 학습에 연결시키는 과정을 다루는 문제입니다.

## 문제 14. Knowledge Distillation (Soft Targets KL Loss)

In [ ]:
def kd_loss_step(teacher, student, inputs, labels, T,
                 soft_target_loss_weight, ce_loss_weight, ce_loss):
    with torch.no_grad():
        teacher_logits = teacher(inputs)
    student_logits = student(inputs)

    soft_targets = ## Write Answer
    student_prob = ## Write Answer

    soft_targets_loss = ## Write Answer

    label_loss = ce_loss(student_logits, labels)
    loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss
    return loss

## 문제 15. Cosine Loss 기반 KD

In [ ]:
def cosine_kd_step(teacher, student, inputs, labels,
                   hidden_rep_loss_weight, ce_loss_weight, ce_loss, cosine_loss):
    with torch.no_grad():
        _, teacher_hidden = teacher(inputs)
    student_logits, student_hidden = student(inputs)

    hidden_rep_loss = ## Write Answer

    label_loss = ce_loss(student_logits, labels)
    loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss

## 문제 16. MSE Feature Map 기반 KD

In [ ]:
def mse_kd_step(teacher, student, inputs, labels,
                feature_map_weight, ce_loss_weight, ce_loss, mse_loss):
    with torch.no_grad():
        _, teacher_feature_map = teacher(inputs)
    student_logits, regressor_feature_map = student(inputs)

    hidden_rep_loss = ## Write Answer

    label_loss = ce_loss(student_logits, labels)
    loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss